#  This is the bronze layer where we gather the data.  

In our case we have pre-made datasets, these datasets where made by AI.  
The datasets that where created are in blob-format, with the ID, Title, and content sepperated.  


In [3]:
import os
import shutil
import pandas as pd
from pypdf import PdfReader

raw_folder = '../../Data/raw'
bronze_folder = '../../Data/bronze'


def clean_raw_txt_files(folder):
    for f in os.listdir(folder):
        if f.startswith("doc_") and f.endswith(".txt"):
            os.remove(os.path.join(folder, f))


def clean_bronze_folder(folder):
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder)


def get_next_doc_id(counter):
    return f"doc_{counter:02d}"


# Wipe and reset
clean_raw_txt_files(raw_folder)
clean_bronze_folder(bronze_folder)

# Step 1: Convert PDFs to TXT
counter = 1
for filename in sorted(os.listdir(raw_folder)):
    if not filename.lower().endswith('.pdf'):
        continue

    pdf_path = os.path.join(raw_folder, filename)
    document_id = get_next_doc_id(counter)
    txt_path = os.path.join(raw_folder, f'{document_id}.txt')

    try:
        reader = PdfReader(pdf_path, strict=False)
        text = ""

        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"

        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(text)

        print(f"Created {txt_path} from {filename}")
        counter += 1

    except Exception as e:
        print(f"Failed to process {filename}: {e}")


# Step 2: Convert TXT files to Bronze JSON
for filename in sorted(os.listdir(raw_folder)):
    if not filename.lower().endswith('.txt'):
        continue

    document_id = os.path.splitext(filename)[0]
    txt_path = os.path.join(raw_folder, filename)
    bronze_output_path = os.path.join(bronze_folder, f'{document_id}_bronze.json')

    with open(txt_path, 'r', encoding='utf-8') as f:
        text = f.read()

    bronze_data = pd.DataFrame([{
        "document_id": document_id,
        "raw_text": text
    }])

    bronze_data.to_json(bronze_output_path, orient='records', force_ascii=False)
    print(f"Created bronze JSON for {filename}")

Created ../../Data/raw\doc_01.txt from ppdfinal.pdf
Created bronze JSON for doc_01.txt
